# 🎵 EAS 587 – Spotify Analysis | Bronze Layer
**Team TriForce** — Dilip, Harsha, Pavan

Reads the raw Spotify data from the existing **`workspace.default.dataset`**
table (already uploaded via the Databricks UI) and writes it to a managed
Bronze Delta table — no DBFS, no file paths needed.

In [0]:
from pyspark.sql import functions as F

# ── Configuration ──────────────────────────────────────────────────
SOURCE_TABLE = "workspace.default.dataset"   # already exists in your catalog
BRONZE_CAT   = "workspace"
BRONZE_SCH   = "spotify_bronze"
BRONZE_TBL   = "raw_tracks"

In [0]:
# Create the Bronze schema (database) under the workspace catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CAT}.{BRONZE_SCH}")
print(f"✅ Schema '{BRONZE_CAT}.{BRONZE_SCH}' ready")

✅ Schema 'workspace.spotify_bronze' ready


In [0]:
# Read raw data straight from the existing catalog table
df_raw = spark.table(SOURCE_TABLE)

print(f"📄 Raw rows loaded : {df_raw.count():,}")
print(f"📋 Columns         : {len(df_raw.columns)}")
df_raw.printSchema()

📄 Raw rows loaded : 114,000
📋 Columns         : 20
root
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: long (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: long (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: long (nullable = true)
 |-- track_genre: string (nullable = true)



In [0]:
# Add standard medallion metadata columns
df_bronze = (
    df_raw
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source_table", F.lit(SOURCE_TABLE))
)

In [0]:
# Write as a managed Delta table — no external path needed
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_CAT}.{BRONZE_SCH}.{BRONZE_TBL}")
)

print(f"✅ Bronze table '{BRONZE_CAT}.{BRONZE_SCH}.{BRONZE_TBL}' written")

✅ Bronze table 'workspace.spotify_bronze.raw_tracks' written


## ✅ Validation — Bronze table exists

In [0]:
%sql
SHOW TABLES IN workspace.spotify_bronze;

database,tableName,isTemporary
spotify_bronze,raw_tracks,false


In [0]:
%sql
SELECT * FROM workspace.spotify_bronze.raw_tracks LIMIT 5;

track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,_ingestion_ts,_source_table
5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,false,0.676,0.461,1,-6.746,0,0.143,0.0322,1.01E-6,0.358,0.715,87.917,4,acoustic,2026-04-17T00:15:44.952Z,workspace.default.dataset
4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,false,0.42,0.166,1,-17.235,1,0.0763,0.924,5.56E-6,0.101,0.267,77.489,4,acoustic,2026-04-17T00:15:44.952Z,workspace.default.dataset
1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,false,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4,acoustic,2026-04-17T00:15:44.952Z,workspace.default.dataset
6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,false,0.266,0.0596,0,-18.515,1,0.0363,0.905,7.07E-5,0.132,0.143,181.74,3,acoustic,2026-04-17T00:15:44.952Z,workspace.default.dataset
5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,false,0.618,0.443,2,-9.681,1,0.0526,0.469,0.0,0.0829,0.167,119.949,4,acoustic,2026-04-17T00:15:44.952Z,workspace.default.dataset


In [0]:
%sql
SELECT COUNT(*) AS total_raw_rows FROM workspace.spotify_bronze.raw_tracks;

total_raw_rows
114000
